# Symptom-based disease prediction: full reproducible run (Google Colab)

**How to use:** `Runtime > Run all` (about 5-10 minutes on the free CPU runtime). Every number and figure for the paper is produced here, then downloaded as `results.zip` at the end.

**Dataset:** *Disease Prediction Using Machine Learning* on Kaggle (author: kaushil268), 4,920 rows, 132 symptom columns, 41 diseases. The notebook tries Kaggle first, then GitHub copies, then lets you upload the file yourself.

**Read before using it in a paper**
* It is an educational dataset. Its page does not describe it as real patient data, so do not call it that.
* A third party reports that most rows are exact copies (about 304 distinct rows). Cell 4 measures this on your copy, and the pipeline removes duplicates *before* splitting, which is the leakage fix the reviewers asked for. Expect lower and more variable scores than studies that split the raw file.
* If your paper used a different dataset, run the notebook on **that** file instead (upload it in Cell 3) and describe whichever dataset you actually used.

## 1. Install

In [ ]:
!pip -q install kagglehub flask psutil

## 2. Code files (written to the Colab disk)

In [ ]:
%%writefile utils.py
"""Shared helpers: environment capture and latency statistics."""
import importlib.metadata as md
import platform
import subprocess
import sys

import numpy as np


def _cpu_model():
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/cpuinfo") as fh:
                for line in fh:
                    if line.lower().startswith("model name"):
                        return line.split(":", 1)[1].strip()
        if sys.platform == "darwin":
            return subprocess.check_output(["sysctl", "-n", "machdep.cpu.brand_string"], text=True).strip()
        if sys.platform.startswith("win"):
            out = subprocess.check_output(
                ["powershell", "-NoProfile", "-Command", "(Get-CimInstance Win32_Processor).Name"],
                text=True, stderr=subprocess.DEVNULL).strip()
            if out:
                return out.splitlines()[0].strip()
    except Exception:
        pass
    return platform.processor() or "unknown"


def _ram_gb():
    try:
        import psutil
        return round(psutil.virtual_memory().total / 1024 ** 3, 1)
    except Exception:
        pass
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/meminfo") as fh:
                for line in fh:
                    if line.startswith("MemTotal"):
                        return round(int(line.split()[1]) / 1024 ** 2, 1)
    except Exception:
        pass
    return None


def _ver(pkg):
    try:
        return md.version(pkg)
    except Exception:
        return None


def env_info():
    return {
        "python": sys.version.split()[0],
        "os": platform.platform(),
        "cpu_model": _cpu_model(),
        "logical_cores": __import__("os").cpu_count(),
        "ram_gb": _ram_gb(),
        "packages": {p: _ver(p) for p in
                     ["numpy", "pandas", "scikit-learn", "joblib", "matplotlib", "flask"]},
    }


def latency_stats(ms):
    a = np.asarray(ms, dtype=float)
    return {
        "n": int(a.size), "mean_ms": float(a.mean()), "median_ms": float(np.median(a)),
        "std_ms": float(a.std(ddof=1)) if a.size > 1 else 0.0,
        "p95_ms": float(np.percentile(a, 95)), "p99_ms": float(np.percentile(a, 99)),
        "min_ms": float(a.min()), "max_ms": float(a.max()),
    }

In [ ]:
%%writefile train_evaluate.py
#!/usr/bin/env python3
"""
Reproducible training and evaluation for symptom-based disease prediction.

Input : a CSV with one row per record, one 0/1 column per symptom, and one label column.
Output: <out>/results.json plus tables (CSV), figures (PNG, 300 dpi) and model.joblib.

Protocol (every step is enforced or verified by this script):
  1. Exact duplicate (symptom vector + label) rows are removed BEFORE splitting.
  2. One stratified train/test split with a fixed seed. The test set is used once, for final scoring.
  3. All hyper-parameter tuning uses stratified k-fold cross-validation on the TRAINING part only.
  4. No fitted preprocessing is used (binary features are used as-is), so nothing is learned from test rows.
  5. The same split, folds and scoring rule are used for all four classifiers.
  6. The split is repeated with other seeds (full tuning each time) to report run-to-run variation.
"""
import argparse
import json
import os
import random
import time
import warnings
from pathlib import Path

os.environ.setdefault("PYTHONWARNINGS", "ignore::FutureWarning")
warnings.filterwarnings("ignore", category=FutureWarning)

import joblib
import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
from sklearn.ensemble import RandomForestClassifier  # noqa: E402
from sklearn.inspection import permutation_importance  # noqa: E402
from sklearn.metrics import (accuracy_score, confusion_matrix,  # noqa: E402
                             precision_recall_fscore_support, roc_auc_score, roc_curve)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split  # noqa: E402
from sklearn.naive_bayes import BernoulliNB  # noqa: E402
from sklearn.preprocessing import LabelEncoder  # noqa: E402
from sklearn.svm import SVC  # noqa: E402
from sklearn.tree import DecisionTreeClassifier  # noqa: E402

from utils import env_info, latency_stats  # noqa: E402

GRIDS = {
    "Random Forest": (lambda s: RandomForestClassifier(random_state=s, n_jobs=-1),
                      {"n_estimators": [100, 200, 300], "max_depth": [None, 10, 20],
                       "min_samples_leaf": [1, 2, 4]}),
    "Naive Bayes": (lambda s: BernoulliNB(), {"alpha": [0.1, 0.5, 1.0]}),
    "SVM": (lambda s: SVC(kernel="rbf", probability=True, random_state=s), {"C": [0.1, 1, 10]}),
    "Decision Tree": (lambda s: DecisionTreeClassifier(random_state=s),
                      {"max_depth": [None, 10, 20], "min_samples_leaf": [1, 2, 4]}),
}


def jsonable(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    raise TypeError(type(o))


# ----------------------------------------------------------------------------- data
def load_and_clean(args):
    df = pd.read_csv(args.data)
    if args.label_col not in df.columns:
        raise SystemExit(f"Label column '{args.label_col}' not found. Columns: {list(df.columns)[:10]}...")
    drop = [c for c in (args.drop_cols or []) if c in df.columns]
    feats = [c for c in df.columns if c != args.label_col and c not in drop]
    X = df[feats].apply(pd.to_numeric, errors="coerce")
    n_missing = int(X.isna().sum().sum())
    if n_missing and not args.fill_missing_zero:
        raise SystemExit(f"{n_missing} missing/non-numeric symptom cells. Clean them, or pass "
                         "--fill-missing-zero to treat them as 'symptom not reported' (0).")
    X = X.fillna(0)
    if not set(np.unique(X.values)) <= {0, 1, 0.0, 1.0}:
        raise SystemExit("Symptom columns must contain only 0/1 values.")
    X = X.astype(np.uint8)
    y_raw = df[args.label_col].astype(str).str.strip()
    info = {"raw_records": int(len(df)), "n_features": int(len(feats)),
            "missing_cells_filled_with_zero": n_missing if args.fill_missing_zero else 0,
            "dropped_columns": drop}

    full = X.copy()
    full["__label__"] = y_raw.values
    exact = full.duplicated(keep="first")
    info["exact_duplicate_rows_removed"] = int(exact.sum())
    X, y_raw = X[~exact.values], y_raw[~exact.values]

    h = pd.util.hash_pandas_object(X, index=False)
    nun = pd.DataFrame({"h": h.values, "y": y_raw.values}).groupby("h")["y"].nunique()
    conflict_hashes = set(nun[nun > 1].index)
    conflict_mask = h.isin(conflict_hashes).values
    info["rows_with_identical_symptoms_but_different_labels"] = int(conflict_mask.sum())
    info["conflicting_rows_dropped"] = bool(args.drop_conflicts)
    if args.drop_conflicts:
        X, y_raw = X[~conflict_mask], y_raw[~conflict_mask]
    info["records_after_cleaning"] = int(len(X))
    info["unique_symptom_vectors"] = int(pd.util.hash_pandas_object(X, index=False).nunique())

    le = LabelEncoder()
    y = le.fit_transform(y_raw.values)
    classes = [str(c) for c in le.classes_]
    info["n_classes"] = len(classes)
    info["class_counts"] = {c: int((y == i).sum()) for i, c in enumerate(classes)}
    info["symptoms_per_record_mean"] = float(X.values.sum(axis=1).mean())
    return X.values.astype(np.float32), y, feats, classes, info


# ----------------------------------------------------------------------------- metrics
def auc_macro(y, proba, n_classes):
    if n_classes == 2:
        return float(roc_auc_score(y, proba[:, 1])) if len(set(y)) == 2 else None
    vals = []
    for c in range(n_classes):
        pos = (y == c)
        if 0 < pos.sum() < len(y):
            vals.append(roc_auc_score(pos, proba[:, c]))
    return float(np.mean(vals)) if vals else None


def calibration(y, proba, n_bins=10):
    conf, pred = proba.max(axis=1), proba.argmax(axis=1)
    correct = (pred == y).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    rows, ece = [], 0.0
    for b in range(n_bins):
        lo, hi = edges[b], edges[b + 1]
        m = (conf > lo) & (conf <= hi) if b else (conf >= lo) & (conf <= hi)
        if m.sum():
            gap = abs(correct[m].mean() - conf[m].mean())
            ece += m.sum() / len(y) * gap
            rows.append({"bin": f"({lo:.1f},{hi:.1f}]", "n": int(m.sum()),
                         "mean_confidence": float(conf[m].mean()), "accuracy": float(correct[m].mean())})
    onehot = np.eye(proba.shape[1])[y]
    brier = float(((proba - onehot) ** 2).sum(axis=1).mean())
    return {"ece_10bins": float(ece), "brier_multiclass": brier}, rows


def metrics(y, pred, proba, classes):
    n = len(classes)
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=list(range(n)), average="macro", zero_division=0)
    _, _, wf, _ = precision_recall_fscore_support(y, pred, labels=list(range(n)), average="weighted", zero_division=0)
    cal, _ = calibration(y, proba)
    return {"accuracy": float(accuracy_score(y, pred)), "precision_macro": float(p), "recall_macro": float(r),
            "f1_macro": float(f), "f1_weighted": float(wf), "auc_macro_ovr": auc_macro(y, proba, n), **cal}


def fit_all(Xtr, ytr, Xte, yte, classes, seed, cv, jobs):
    out = {}
    cv = min(cv, int(np.bincount(ytr).min()))
    if cv < 2:
        raise SystemExit("A class has fewer than 2 training records after splitting; cross-validation is impossible. "
                         "Use more data, fewer classes, or a smaller --test-size.")
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=seed)
    for name, (make, grid) in GRIDS.items():
        gs = GridSearchCV(make(seed), grid, scoring="f1_macro", cv=skf, n_jobs=jobs, refit=True)
        gs.fit(Xtr, ytr)
        est = gs.best_estimator_
        out[name] = {"estimator": est, "best_params": gs.best_params_,
                     "cv_f1_macro_mean": float(gs.best_score_),
                     "cv_f1_macro_std": float(gs.cv_results_["std_test_score"][gs.best_index_]),
                     "test": metrics(yte, est.predict(Xte), est.predict_proba(Xte), classes)}
    return out


# ----------------------------------------------------------------------------- figures
def save_fig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=300)
    plt.close()


def fig_class_distribution(info, out):
    names, counts = list(info["class_counts"]), list(info["class_counts"].values())
    k = len(names)
    if k > 12:
        order = np.argsort(counts)
        plt.figure(figsize=(3.4, max(3, 0.12 * k)))
        plt.barh(np.array(names)[order], np.array(counts)[order], color="#1f5f8b")
        plt.yticks(fontsize=5)
        plt.xlabel("Records")
    else:
        plt.figure(figsize=(3.4, 2.6))
        plt.bar(names, counts, color="#1f5f8b")
        plt.xticks(rotation=30 if k > 3 else 0, ha="right" if k > 3 else "center", fontsize=7)
        plt.ylabel("Records")
    save_fig(out / "class_distribution.png")


def fig_confusion(cm, classes, out, normalized=False):
    k = len(classes)
    data = cm / cm.sum(axis=1, keepdims=True).clip(min=1) if normalized else cm
    size = 3.4 if k <= 6 else min(7, 0.3 * k + 2)
    plt.figure(figsize=(size, size * 0.9))
    plt.imshow(data, cmap="Blues", vmin=0, vmax=1 if normalized else None)
    plt.colorbar(fraction=0.046, pad=0.04)
    fs = 8 if k <= 6 else (6 if k <= 10 else 4)
    plt.xticks(range(k), classes, rotation=45, ha="right", fontsize=fs if k <= 15 else 3)
    plt.yticks(range(k), classes, fontsize=fs if k <= 15 else 3)
    plt.xlabel("Predicted class", fontsize=8)
    plt.ylabel("Actual class", fontsize=8)
    if k <= 15:
        for i in range(k):
            for j in range(k):
                v = data[i, j]
                txt = f"{v:.2f}" if normalized else f"{int(v)}"
                plt.text(j, i, txt, ha="center", va="center", fontsize=fs,
                         color="white" if v > data.max() * 0.6 else "black")
    save_fig(out / ("confusion_matrix_normalized.png" if normalized else "confusion_matrix.png"))


def fig_roc(y, proba, classes, out):
    n = len(classes)
    plt.figure(figsize=(3.4, 3))
    if n == 2:
        fpr, tpr, _ = roc_curve(y, proba[:, 1])
        plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y, proba[:, 1]):.3f}")
    else:
        grid = np.linspace(0, 1, 200)
        tprs = []
        for c in range(n):
            pos = (y == c)
            if 0 < pos.sum() < len(y):
                fpr, tpr, _ = roc_curve(pos, proba[:, c])
                tprs.append(np.interp(grid, fpr, tpr))
        plt.plot(grid, np.mean(tprs, axis=0), label=f"macro one-vs-rest, AUC = {auc_macro(y, proba, n):.3f}")
    plt.plot([0, 1], [0, 1], "k--", lw=0.8)
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.legend(fontsize=7)
    save_fig(out / "roc_curve.png")


def fig_reliability(rows, out):
    plt.figure(figsize=(3.4, 3))
    plt.plot([0, 1], [0, 1], "k--", lw=0.8, label="Perfect calibration")
    plt.plot([r["mean_confidence"] for r in rows], [r["accuracy"] for r in rows], "o-", color="#1f5f8b",
             label="Random Forest")
    plt.xlabel("Mean predicted confidence")
    plt.ylabel("Observed accuracy")
    plt.legend(fontsize=7)
    save_fig(out / "reliability_diagram.png")


def oob_curve(params, Xtr, ytr, seed, out, jobs):
    n_final = int(params.get("n_estimators", 100))
    step = max(10, n_final // 20)
    ns = list(range(step, n_final + 1, step))
    if ns[-1] != n_final:
        ns.append(n_final)
    rf = RandomForestClassifier(**{**params, "n_estimators": ns[0]}, oob_score=True, warm_start=True,
                                random_state=seed, n_jobs=jobs, bootstrap=True)
    rows = []
    for n in ns:
        rf.set_params(n_estimators=n)
        rf.fit(Xtr, ytr)
        rows.append({"n_estimators": n, "oob_error": float(1 - rf.oob_score_)})
    pd.DataFrame(rows).to_csv(out / "oob_error.csv", index=False)
    plt.figure(figsize=(3.4, 2.6))
    plt.plot([r["n_estimators"] for r in rows], [r["oob_error"] for r in rows], "o-", ms=3, color="#1f5f8b")
    plt.xlabel("Number of trees")
    plt.ylabel("Out-of-bag error")
    save_fig(out / "oob_error.png")
    return rows


def importance(rf, Xte, yte, feats, seed, out, jobs):
    pi = permutation_importance(rf, Xte, yte, scoring="accuracy", n_repeats=10, random_state=seed, n_jobs=jobs)
    df = pd.DataFrame({"feature": feats, "gini_importance": rf.feature_importances_,
                       "permutation_mean": pi.importances_mean, "permutation_std": pi.importances_std})
    df["gini_rank"] = df["gini_importance"].rank(ascending=False, method="min").astype(int)
    df["permutation_rank"] = df["permutation_mean"].rank(ascending=False, method="min").astype(int)
    df = df.sort_values("gini_importance", ascending=False)
    df.to_csv(out / "feature_importance.csv", index=False)
    top = df.sort_values("permutation_mean", ascending=False).head(12)[::-1]
    plt.figure(figsize=(3.4, 3.2))
    plt.barh(top["feature"], top["permutation_mean"], xerr=top["permutation_std"], color="#1f5f8b")
    plt.xlabel("Drop in test accuracy when shuffled", fontsize=7)
    plt.yticks(fontsize=6)
    save_fig(out / "feature_importance.png")
    return df


def timing(models, Xte, seed, n=1000, warmup=50):
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(Xte), size=min(n, len(Xte)), replace=len(Xte) < n)
    res = {}
    for name, est in models.items():
        if hasattr(est, "n_jobs"):
            est.set_params(n_jobs=1)
        for i in idx[:warmup]:
            est.predict_proba(Xte[i:i + 1])
        lat = []
        for i in idx:
            t = time.perf_counter()
            est.predict_proba(Xte[i:i + 1])
            lat.append((time.perf_counter() - t) * 1e3)
        res[name] = latency_stats(lat)
    return res


# ----------------------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data", required=True)
    ap.add_argument("--label-col", required=True)
    ap.add_argument("--drop-cols", nargs="*", help="non-symptom columns to ignore (ids etc.)")
    ap.add_argument("--out", default="results")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--test-size", type=float, default=0.2)
    ap.add_argument("--cv", type=int, default=5)
    ap.add_argument("--repeats", type=int, default=5, help="extra full re-runs with other seeds (0 to skip)")
    ap.add_argument("--jobs", type=int, default=-1)
    ap.add_argument("--fill-missing-zero", action="store_true")
    ap.add_argument("--drop-conflicts", action="store_true",
                    help="drop rows whose symptom vector appears with more than one label")
    args = ap.parse_args()

    random.seed(args.seed)
    np.random.seed(args.seed)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)

    X, y, feats, classes, data_info = load_and_clean(args)
    n_classes = len(classes)
    min_count = min(data_info["class_counts"].values())
    cv = args.cv
    if min_count < 2:
        raise SystemExit("A class has fewer than 2 records; stratified splitting is impossible.")

    idx = np.arange(len(y))
    Xtr, Xte, ytr, yte, itr, ite = train_test_split(X, y, idx, test_size=args.test_size, stratify=y,
                                                    random_state=args.seed)
    cv = min(cv, int(np.bincount(ytr).min()))
    htr = pd.util.hash_pandas_object(pd.DataFrame(Xtr), index=False).values
    hte = pd.util.hash_pandas_object(pd.DataFrame(Xte), index=False).values
    overlap_rows = int(np.isin(hte, htr).sum())
    split_info = {"seed": args.seed, "test_size": args.test_size, "stratified": True,
                  "n_train": int(len(ytr)), "n_test": int(len(yte)), "cv_folds": int(cv),
                  "cv_used_for_tuning_on": "training part only", "tuning_scoring": "f1_macro",
                  "test_rows_whose_symptom_vector_also_occurs_in_train": overlap_rows,
                  "train_class_counts": {c: int((ytr == i).sum()) for i, c in enumerate(classes)},
                  "test_class_counts": {c: int((yte == i).sum()) for i, c in enumerate(classes)}}
    json.dump({"train": itr.tolist(), "test": ite.tolist()}, open(out / "split_indices.json", "w"))

    print("Tuning and fitting 4 classifiers on the training split ...")
    fits = fit_all(Xtr, ytr, Xte, yte, classes, args.seed, cv, args.jobs)
    rf = fits["Random Forest"]["estimator"]
    proba, pred = rf.predict_proba(Xte), rf.predict(Xte)

    # per-class table and confusion matrix (Random Forest)
    p, r, f, s = precision_recall_fscore_support(yte, pred, labels=list(range(n_classes)), zero_division=0)
    pd.DataFrame({"class": classes, "precision": p, "recall": r, "f1": f, "support": s}).to_csv(
        out / "per_class_metrics_random_forest.csv", index=False)
    cm = confusion_matrix(yte, pred, labels=list(range(n_classes)))
    pd.DataFrame(cm, index=classes, columns=classes).to_csv(out / "confusion_matrix_random_forest.csv")
    pairs = [{"actual": classes[i], "predicted": classes[j], "count": int(cm[i, j]),
              "share_of_actual_class": float(cm[i, j] / max(cm[i].sum(), 1))}
             for i in range(n_classes) for j in range(n_classes) if i != j and cm[i, j] > 0]
    pairs = sorted(pairs, key=lambda d: -d["count"])[:15]
    pd.DataFrame(pairs).to_csv(out / "top_confusions_random_forest.csv", index=False)

    fig_class_distribution(data_info, out)
    fig_confusion(cm, classes, out)
    fig_confusion(cm, classes, out, normalized=True)
    fig_roc(yte, proba, classes, out)
    cal_summary, cal_rows = calibration(yte, proba)
    pd.DataFrame(cal_rows).to_csv(out / "calibration_bins.csv", index=False)
    fig_reliability(cal_rows, out)
    oob_rows = oob_curve(fits["Random Forest"]["best_params"], Xtr, ytr, args.seed, out, args.jobs)
    imp = importance(rf, Xte, yte, feats, args.seed, out, args.jobs)

    print("Timing single-record inference (model only, n_jobs=1) ...")
    lat = timing({k: v["estimator"] for k, v in fits.items()}, Xte, args.seed)

    repeats = {}
    if args.repeats > 0:
        print(f"Repeating the whole procedure with {args.repeats} other seeds ...")
        acc = {k: [] for k in GRIDS}
        f1s = {k: [] for k in GRIDS}
        aucs = {k: [] for k in GRIDS}
        for k in range(1, args.repeats + 1):
            s = args.seed + k
            a, b, c, d = train_test_split(X, y, test_size=args.test_size, stratify=y, random_state=s)
            res = fit_all(a, c, b, d, classes, s, cv, args.jobs)
            for name, v in res.items():
                acc[name].append(v["test"]["accuracy"])
                f1s[name].append(v["test"]["f1_macro"])
                aucs[name].append(v["test"]["auc_macro_ovr"])
        for name in GRIDS:
            av = [x for x in aucs[name] if x is not None]
            repeats[name] = {"n_repeats": args.repeats,
                             "accuracy_mean": float(np.mean(acc[name])), "accuracy_std": float(np.std(acc[name], ddof=1)) if args.repeats > 1 else 0.0,
                             "f1_macro_mean": float(np.mean(f1s[name])), "f1_macro_std": float(np.std(f1s[name], ddof=1)) if args.repeats > 1 else 0.0,
                             "auc_mean": float(np.mean(av)) if av else None,
                             "auc_std": float(np.std(av, ddof=1)) if len(av) > 1 else 0.0}

    models_out = {k: {kk: vv for kk, vv in v.items() if kk != "estimator"} for k, v in fits.items()}
    results = {
        "data": data_info, "split": split_info, "grids": {k: v[1] for k, v in GRIDS.items()},
        "models": models_out, "repeated_splits": repeats,
        "random_forest": {
            "positive_class_for_binary_auc": classes[1] if n_classes == 2 else None,
            "n_features_used_by_trees": int(rf.n_features_in_), "n_trees": int(rf.n_estimators),
            "calibration": cal_summary, "oob_error_final": oob_rows[-1]["oob_error"], "oob_first": oob_rows[0], "oob_last": oob_rows[-1],
            "top_gini": imp.head(10)[["feature", "gini_importance", "permutation_mean", "permutation_std"]].to_dict("records"),
            "top_confusions": pairs[:5]},
        "latency_model_only": lat,
        "leakage_checks": {
            "exact_duplicates_removed_before_split": True,
            "test_rows_sharing_a_symptom_vector_with_train": overlap_rows,
            "hyperparameters_tuned_on_training_part_only": True,
            "fitted_preprocessing_steps": "none (binary features used unchanged)",
            "test_set_used_for": "final scoring, permutation importance and calibration only"},
        "environment": env_info(),
        "classes": classes, "feature_names": feats,
    }
    json.dump(results, open(out / "results.json", "w"), indent=2, default=jsonable)
    json.dump(feats, open(out / "feature_names.json", "w"))
    json.dump(classes, open(out / "classes.json", "w"))
    joblib.dump({"model": rf, "features": feats, "classes": classes}, out / "model.joblib")

    print("\n=== Test-set results (single split) ===")
    for k, v in models_out.items():
        t = v["test"]
        print(f"{k:14s} acc={t['accuracy']:.3f} P={t['precision_macro']:.3f} R={t['recall_macro']:.3f} "
              f"F1={t['f1_macro']:.3f} AUC={t['auc_macro_ovr']}")
    print(f"\nTrain/test vector overlap: {overlap_rows} rows. Everything saved in: {out.resolve()}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile app.py
"""Minimal REST API around the trained model (Flask).

Run:   python app.py --model results/model.joblib
Query: curl -X POST http://127.0.0.1:5000/predict -H "Content-Type: application/json" \
            -d '{"symptoms": ["fever", "cough"]}'
"""
import argparse
import os

import joblib
import numpy as np
from flask import Flask, jsonify, request

MODEL_PATH = os.environ.get("MODEL_PATH", "results/model.joblib")
_bundle = {}


def get_bundle():
    if not _bundle:
        b = joblib.load(MODEL_PATH)
        b["model"].set_params(n_jobs=1)
        b["index"] = {f.strip().lower().replace(" ", "_"): i for i, f in enumerate(b["features"])}
        _bundle.update(b)
    return _bundle


def create_app():
    app = Flask(__name__)

    @app.get("/health")
    def health():
        b = get_bundle()
        return jsonify(status="ok", n_symptoms=len(b["features"]), n_classes=len(b["classes"]))

    @app.get("/symptoms")
    def symptoms():
        return jsonify(symptoms=get_bundle()["features"])

    @app.post("/predict")
    def predict():
        b = get_bundle()
        data = request.get_json(silent=True) or {}
        names = data.get("symptoms")
        if not isinstance(names, list) or not names:
            return jsonify(error="Send JSON like {\"symptoms\": [\"fever\", \"cough\"]}"), 400
        x = np.zeros((1, len(b["features"])), dtype=np.float32)
        unknown = []
        for s in names:
            key = str(s).strip().lower().replace(" ", "_")
            if key in b["index"]:
                x[0, b["index"][key]] = 1.0
            else:
                unknown.append(s)
        if unknown:
            return jsonify(error="Unknown symptoms", unknown=unknown), 400
        proba = b["model"].predict_proba(x)[0]
        top = np.argsort(proba)[::-1][:3]
        return jsonify(prediction=b["classes"][int(top[0])], confidence=float(proba[top[0]]),
                       top3=[{"class": b["classes"][int(i)], "probability": float(proba[i])} for i in top],
                       n_symptoms=int(x.sum()),
                       note="Screening aid only; not a medical diagnosis.")

    return app


app = create_app()

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default=MODEL_PATH)
    ap.add_argument("--port", type=int, default=5000)
    a = ap.parse_args()
    os.environ["MODEL_PATH"] = a.model
    MODEL_PATH = a.model
    app.run(host="127.0.0.1", port=a.port)

In [ ]:
%%writefile benchmark_api.py
"""End-to-end latency of the /predict endpoint.

Default mode runs the Flask app in-process (test client): includes JSON parsing, validation, vector
construction, model inference and JSON serialisation, but not network transfer.
With --url it sends real HTTP requests to a running server (python app.py) and measures the round trip.
Results are written to <out>/api_benchmark.json together with the hardware/software description.
"""
import argparse
import json
import os
import time
import urllib.request
from pathlib import Path

import numpy as np

from utils import env_info, latency_stats


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="results/model.joblib")
    ap.add_argument("--out", default="results")
    ap.add_argument("--n", type=int, default=1000)
    ap.add_argument("--warmup", type=int, default=50)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--tag", default="", help="suffix for the output file name, e.g. http")
    ap.add_argument("--url", help="e.g. http://127.0.0.1:5000/predict (server must already be running)")
    a = ap.parse_args()

    os.environ["MODEL_PATH"] = a.model
    import app as appmod
    appmod.MODEL_PATH = a.model
    feats = appmod.get_bundle()["features"]
    rng = np.random.default_rng(a.seed)
    payloads = [json.dumps({"symptoms": list(rng.choice(feats, size=int(rng.integers(3, 9)), replace=False))})
                for _ in range(a.n + a.warmup)]

    if a.url:
        def send(p):
            req = urllib.request.Request(a.url, data=p.encode(), headers={"Content-Type": "application/json"})
            with urllib.request.urlopen(req) as r:
                return r.status
        mode = "http_round_trip_localhost"
    else:
        client = appmod.app.test_client()

        def send(p):
            return client.post("/predict", data=p, content_type="application/json").status_code
        mode = "in_process_flask_test_client"

    lat, bad = [], 0
    for i, p in enumerate(payloads):
        t = time.perf_counter()
        status = send(p)
        dt = (time.perf_counter() - t) * 1e3
        if i >= a.warmup:
            lat.append(dt)
            bad += status != 200
    res = {"mode": mode, "requests": a.n, "non_200_responses": bad,
           "symptoms_per_request": "random 3-8", "latency": latency_stats(lat), "environment": env_info()}
    Path(a.out).mkdir(exist_ok=True)
    fname = "api_benchmark" + ("_" + a.tag if a.tag else "") + ".json"
    json.dump(res, open(Path(a.out) / fname, "w"), indent=2)
    print(json.dumps(res["latency"], indent=2))


if __name__ == "__main__":
    main()

## 3. Download the dataset and convert it to one 0/1 column per symptom

In [ ]:
import os, re, json, datetime, subprocess, sys, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
os.makedirs("data", exist_ok=True)

KAGGLE_SLUG = "kaushil268/disease-prediction-using-machine-learning"
KAGGLE_URL = "https://www.kaggle.com/datasets/" + KAGGLE_SLUG
CANDIDATE_URLS = [   # tried in order if the Kaggle download fails; both layouts are understood
    "https://raw.githubusercontent.com/Sarvan-Singh7/Desease-Prediction-System-using-ML/main/Data/Training.csv",
    "https://raw.githubusercontent.com/Sarvan-Singh7/Desease-Prediction-System-using-ML/master/Data/Training.csv",
    "https://raw.githubusercontent.com/yash-naikk/HEALTH-CARE-CHATBOT/main/dataset.csv",
]

def tidy(s):
    """'dischromic _patches' / ' spotting_ urination' -> 'dischromic_patches' / 'spotting_urination'"""
    return re.sub(r"\s+", "_", re.sub(r"\s*_\s*", "_", str(s).strip()))

PREP = {}
def to_wide(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    low = {c.lower(): c for c in df.columns}
    if "prognosis" in low:                                   # layout A: one 0/1 column per symptom
        lab = low["prognosis"]
        df = df[[c for c in df.columns if not c.lower().startswith("unnamed")]]
        X = df[[c for c in df.columns if c != lab]].apply(pd.to_numeric, errors="coerce")
        PREP["missing_cells_filled_with_zero"] = int(X.isna().sum().sum())
        out = X.fillna(0).astype(int)
        out.columns = [tidy(c) for c in out.columns]
        out["prognosis"] = df[lab].astype(str).str.strip().values
        PREP["layout"] = "wide (Training.csv style)"
        return out
    # layout B: disease in the first column, then up to ~17 symptom-name columns
    if df.columns[0].lower() not in ("disease", "label", "diagnosis"):   # file had no header row
        df = pd.concat([pd.DataFrame([list(df.columns)], columns=df.columns), df], ignore_index=True)
    recs = []
    for _, r in df.iterrows():
        syms = sorted({tidy(x) for x in r.iloc[1:] if pd.notna(x) and str(x).strip() and not str(x).startswith("Unnamed:")})
        recs.append((str(r.iloc[0]).strip(), syms))
    vocab = sorted({s for _, ss in recs for s in ss}); pos = {s: j for j, s in enumerate(vocab)}
    arr = np.zeros((len(recs), len(vocab)), dtype=int)
    for i, (_, ss) in enumerate(recs):
        for s in ss: arr[i, pos[s]] = 1
    out = pd.DataFrame(arr, columns=vocab); out["prognosis"] = [l for l, _ in recs]
    PREP["layout"] = "long (disease + symptom names), converted to multi-hot"; PREP["missing_cells_filled_with_zero"] = 0
    return out

def from_kaggle():
    import kagglehub
    folder = kagglehub.dataset_download(KAGGLE_SLUG)
    for root, _, files_ in os.walk(folder):
        for f in files_:
            if f.lower() == "training.csv": return pd.read_csv(os.path.join(root, f)), KAGGLE_URL + " (via kagglehub)"
    raise FileNotFoundError("Training.csv not found in the Kaggle download")

loaders = [from_kaggle] + [(lambda u=u: (pd.read_csv(u), u)) for u in CANDIDATE_URLS]
raw, SOURCE_USED = None, None
for fn in loaders:
    try:
        raw, SOURCE_USED = fn(); wide = to_wide(raw)
        if wide.shape[0] > 100 and wide.shape[1] > 20: break
    except Exception as e:
        print("source failed:", type(e).__name__, str(e)[:120]); raw = None
if raw is None:
    print("All automatic sources failed. Upload the CSV (Kaggle Training.csv or dataset.csv, or your own file).")
    from google.colab import files
    up = files.upload(); name = next(iter(up)); raw, SOURCE_USED = pd.read_csv(name), "uploaded file: " + name
    wide = to_wide(raw)

zero_cols = [c for c in wide.columns if c != "prognosis" and wide[c].sum() == 0]
wide = wide.drop(columns=zero_cols)
PREP.update(source=SOURCE_USED, rows=int(len(wide)), symptoms=int(wide.shape[1] - 1), diseases=int(wide["prognosis"].nunique()),
            all_zero_symptom_columns_removed=zero_cols)
wide.to_csv("data/symptoms.csv", index=False)
print(json.dumps(PREP, indent=1))
cnt = wide["prognosis"].value_counts(); print("records per disease: min", cnt.min(), "max", cnt.max())

## 4. How much leakage would a naive split cause? (evidence for the paper)
This counts test rows that are exact copies (same symptoms, same disease) of a training row when the *raw* file is split at random. The pipeline below removes such duplicates before splitting.

In [ ]:
from sklearn.model_selection import train_test_split
w = pd.read_csv("data/symptoms.csv")
feats = [c for c in w.columns if c != "prognosis"]
h_full = pd.util.hash_pandas_object(w, index=False).values
LEAK = {"raw_rows": len(w), "exact_duplicate_rows": int(w.duplicated().sum()),
        "distinct_rows": int((~w.duplicated()).sum())}
shares = []
for seed in range(5):
    tr, te = train_test_split(np.arange(len(w)), test_size=0.2, stratify=w["prognosis"], random_state=seed)
    shares.append(np.isin(h_full[te], h_full[tr]).mean())
LEAK["naive_split_test_rows_copied_from_train_mean"] = float(np.mean(shares))
LEAK["naive_split_test_rows_copied_from_train_min_max"] = [float(min(shares)), float(max(shares))]
print(json.dumps(LEAK, indent=1))
json.dump(LEAK, open("data/leakage_demo.json", "w"), indent=1)

## 5. Train, tune, evaluate (all four classifiers, repeated splits)

In [ ]:
!python train_evaluate.py --data data/symptoms.csv --label-col prognosis --out results --seed 42 --repeats 5

## 6. Results: tables

In [ ]:
R = json.load(open("results/results.json"))
names = ["Random Forest", "Naive Bayes", "SVM", "Decision Tree"]
print("DATASET (paper Table II)"); print({k: R["data"][k] for k in ["raw_records","exact_duplicate_rows_removed","records_after_cleaning","unique_symptom_vectors","n_features","n_classes"]})
print("rows with identical symptoms but different disease:", R["data"]["rows_with_identical_symptoms_but_different_labels"])
print("SPLIT:", {k: R["split"][k] for k in ["seed","n_train","n_test","cv_folds","test_rows_whose_symptom_vector_also_occurs_in_train"]})
t5 = pd.DataFrame({n: R["models"][n]["test"] for n in names}).T[["accuracy","precision_macro","recall_macro","f1_macro","auc_macro_ovr","ece_10bins","brier_multiclass"]]
print("\nTABLE V (single split)"); print(t5.round(3))
rows = {n: {"cv_f1": f'{R["models"][n]["cv_f1_macro_mean"]:.3f} +/- {R["models"][n]["cv_f1_macro_std"]:.3f}', **({f'{k}': f'{R["repeated_splits"][n][k+"_mean"]:.3f} +/- {R["repeated_splits"][n][k+"_std"]:.3f}' for k in ["accuracy","f1_macro","auc"]} if R["repeated_splits"] else {})} for n in names}
print("\nTABLE VI (CV and repeated splits)"); print(pd.DataFrame(rows).T)
print("\nTABLE IV (selected hyper-parameters)")
for n in names: print(" ", n, R["models"][n]["best_params"])
print("\nTOP CONFUSIONS (Random Forest)"); print(pd.read_csv("results/top_confusions_random_forest.csv").head(8))
print("\nTABLE VII (feature importance)"); print(pd.read_csv("results/feature_importance.csv").head(10).round(4))
print("\nOOB error:", R["random_forest"]["oob_first"], "->", R["random_forest"]["oob_last"])
print("Calibration (RF):", R["random_forest"]["calibration"])
print("\nTABLE VIII (model-only latency, ms)"); print(pd.DataFrame(R["latency_model_only"]).T.round(3))
print("\nHARDWARE:", R["environment"])

## 7. Results: figures (right-click > save image, or use the zip at the end)

In [ ]:
from IPython.display import Image, display
for f in ["class_distribution","confusion_matrix","confusion_matrix_normalized","roc_curve","oob_error","reliability_diagram","feature_importance"]:
    print(f); display(Image(f"results/{f}.png", width=480))

## 8. API latency (in-process) and real HTTP round trip on localhost

In [ ]:
import time, urllib.request
!python benchmark_api.py --model results/model.joblib --out results --n 500
server = subprocess.Popen([sys.executable, "app.py", "--model", "results/model.joblib", "--port", "5000"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try: urllib.request.urlopen("http://127.0.0.1:5000/health"); break
    except Exception: time.sleep(0.5)
sample = json.dumps({"symptoms": feats[:4]}).encode()
req = urllib.request.Request("http://127.0.0.1:5000/predict", data=sample, headers={"Content-Type": "application/json"})
print("example response:", urllib.request.urlopen(req).read().decode())
subprocess.run([sys.executable, "benchmark_api.py", "--model", "results/model.joblib", "--out", "results", "--n", "500", "--url", "http://127.0.0.1:5000/predict", "--tag", "http"])
server.terminate()
for f in ["api_benchmark", "api_benchmark_http"]:
    a = json.load(open(f"results/{f}.json")); print(f, a["mode"], {k: round(v, 2) for k, v in a["latency"].items()}, "errors:", a["non_200_responses"])

## 9. Dataset description for the paper, and download everything
Edit the empty strings (they stay highlighted in the paper until filled). Use only what the dataset page actually says; if it does not say how labels were produced, write exactly that.

In [ ]:
info = {
  "name": "Disease Prediction Using Machine Learning (Training.csv)",
  "repository": "Kaggle (author: kaushil268)",
  "source_url": KAGGLE_URL,
  "access_date": datetime.date.today().strftime("%d %B %Y"),
  "licence": "",              # copy the licence shown on the Kaggle dataset page
  "record_nature": "",        # e.g. what the page says the records are; do not claim real patient data unless the page does
  "label_generation": "",     # how disease labels were produced, per the page; if undocumented, say so
  "symptom_mapping": "",      # how symptoms map to diseases per the page, and how names were tidied (see PREP above)
  "code_url": "",             # your GitHub repository with these files
  "gap_statement": "",
  "data_ethics": "", "data_protection": "",
}
json.dump(info, open("results/dataset_info.json", "w"), indent=2)
json.dump(PREP, open("results/preparation_log.json", "w"), indent=2, default=str)
import shutil; shutil.copy("data/leakage_demo.json", "results/leakage_demo.json"); shutil.copy("data/symptoms.csv", "results/symptoms_used.csv")
shutil.make_archive("results", "zip", "results")
try:
    from google.colab import files; files.download("results.zip")
except Exception: print("results.zip created in the file browser (left panel)")